In [1]:
#load basic libraries
import polars as pl  # It is advised to use polars, as the detasets can be quite memory-intensive
import numpy as np
import torch.nn as nn
from torch import optim
import torch
import re
from tqdm import trange
import random



%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_direct = "data/BTCUSDT"
X_train = pl.read_parquet(f"{data_direct}/X_train.parquet")
y_train = pl.read_parquet(f"{data_direct}/y_train.parquet")

# load the volume to fill info from the text file
with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
    content = file.read().strip()

match = re.search(r"The volume to fill is: ([\d.]+)", content)
if match:
    volume_to_fill  = float(match.group(1))
    print(f"Extracted number: {volume_to_fill}")
else:
    print("No number found in the file.")


Extracted number: 4.0


# Imputing Data

| Feature Type          | Meaning of NaN                      | Correct Imputation             |
| --------------------- | ----------------------------------- | ------------------------------ |
| OHLC prices (O/H/L/C) | No trade update                     | **Forward-fill**               |
| Volume                | No trade occurred                   | **0**                          |
| LOB prices (bid/ask)  | No new quote → previous still valid | **Forward-fill**               |
| LOB sizes             | No liquidity at that level          | **0**                          |
| First row NaNs        | Missing baseline                    | **Backfill once** |

Since we're dealing with a time series, we're imputing y_train the same way we are X_train.

In [3]:
PRICE_COLS = [
    'ask_price_1','bid_price_1',
    'ask_price_2','bid_price_2',
    'ask_price_3','bid_price_3',
    'ask_price_4','bid_price_4',
    'ask_price_5','bid_price_5',
    'open','high','low','close',
]

VOL_COLS = [
    'ask_vol_1','bid_vol_1',
    'ask_vol_2','bid_vol_2',
    'ask_vol_3','bid_vol_3',
    'ask_vol_4','bid_vol_4',
    'ask_vol_5','bid_vol_5',
    'volume',
]

def backfill_first_nans(df: pl.DataFrame, cols: list[str]):
    for col in cols:
        first_valid_idx = (
            df
            .select((pl.col(col).is_not_null()).arg_max())
            .item()
        )

        if first_valid_idx == 0:
            continue  # no leading NaNs

        # Backfill only the leading NaNs
        df = df.with_columns(
            pl.when(pl.arange(0, pl.len()) < first_valid_idx)
              .then(pl.lit(df[col][first_valid_idx]))
              .otherwise(pl.col(col))
              .alias(col)
        )
    return df


In [4]:
# 0. Sort
X_train = X_train.sort(["anonymized_id", "time_in_hour"])
y_train = y_train.sort(["anonymized_id", "time_in_hour"])

# 1. Backfill *only* leading NaNs (prices only)
X_train = (
    X_train
    .group_by("anonymized_id")
    .map_groups(lambda df: backfill_first_nans(df, PRICE_COLS))
)
y_train = (
    y_train
    .group_by("anonymized_id")
    .map_groups(lambda df: backfill_first_nans(df, PRICE_COLS))
)

# 2. Forward-fill the rest of price fields
X_train = (
    X_train
    .group_by("anonymized_id")
    .map_groups(lambda df: df.with_columns(
        pl.col(PRICE_COLS).fill_null(strategy="forward")
    ))
)
y_train = (
    y_train
    .group_by("anonymized_id")
    .map_groups(lambda df: df.with_columns(
        pl.col(PRICE_COLS).fill_null(strategy="forward")
    ))
)

# 3. Volume columns → 0
X_train = X_train.with_columns(
    pl.col(VOL_COLS).fill_null(0)
)
y_train = y_train.with_columns(
    pl.col(VOL_COLS).fill_null(0)
)


# Cleaning

In [5]:
def df_to_tensor(df: pl.DataFrame, seq_len: int = 60, id_col: str = "anonymized_id", time_col: str = "time_in_hour") -> torch.Tensor:
    """
    Convert a Polars DataFrame into a tensor of shape (seq_len, num_ids, n_features).
    
    Args:
        df: Polars DataFrame containing ID, time, and feature columns.
        seq_len: Number of timesteps per sequence.
        id_col: Column name for the unique ID.
        time_col: Column name for the time dimension.
        
    Returns:
        X: Tensor of shape (seq_len, num_ids, n_features)
    """
    
    # Convert time column to int32
    df = df.with_columns([
        pl.col(time_col).cast(pl.Int32).alias(f"{time_col}_int")
    ])
    
    # Define feature columns
    time_int_col = f"{time_col}_int"
    feature_cols = [col for col in df.columns if col not in [id_col, time_col, time_int_col]]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Convert to tensors
    features = torch.tensor(df.select(feature_cols).to_numpy(), dtype=torch.float32).to(device)
    ids = torch.tensor(df[id_col].to_numpy(), dtype=torch.int64).to(device)
    times = torch.tensor(df[time_int_col].to_numpy()).to(device)
    
    # Unique IDs
    unique_ids = torch.unique(ids)
    num_ids = len(unique_ids)
    n_features = features.shape[1]
    
    # Initialize tensor
    X = torch.zeros(seq_len, num_ids, n_features, dtype=torch.float32).to(device)
    
    for i, uid in enumerate(unique_ids):
        mask = ids == uid
        uid_features = features[mask]
        
        # Sort by time_in_hour if needed
        sorted_idx = torch.argsort(times[mask])
        uid_features = uid_features[sorted_idx]
        
        # Assign to tensor
        X[:, i, :] = uid_features  # assumes exactly seq_len timesteps per ID
        
    return X

# drop time_in_hour within anonymized_id
# Count rows per ID and time_in_hour (duplicates)
dup_ids = (
    X_train
    .group_by(["anonymized_id", "time_in_hour"])
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") > 1)       # only keep duplicates
    .select("anonymized_id")
    .unique()
)

# Drop IDs with missing timesteps
seq_len = 3600-60  # your expected sequence length

# Count timesteps per ID
valid_ids = (
    X_train
    .group_by("anonymized_id")
    .agg(pl.count("time_in_hour").alias("n_timesteps"))
    .filter(pl.col("n_timesteps") == seq_len)
    .select("anonymized_id")
)

# Remove duplicate ids and keep only IDs with full sequences -> 674 ids should be left
X_train = X_train.filter(~pl.col("anonymized_id").is_in(dup_ids["anonymized_id"])).filter(pl.col("anonymized_id").is_in(valid_ids["anonymized_id"]))
y_train = y_train.filter(~pl.col("anonymized_id").is_in(dup_ids["anonymized_id"])).filter(pl.col("anonymized_id").is_in(valid_ids["anonymized_id"]))


y_train = df_to_tensor(y_train, seq_len=60, id_col="anonymized_id", time_col="time_in_hour").to(device)
X_train = df_to_tensor(X_train, seq_len=3600-60, id_col="anonymized_id", time_col="time_in_hour").to(device)


/tmp/ipykernel_15057/2419724157.py:76: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  X_train = X_train.filter(~pl.col("anonymized_id").is_in(dup_ids["anonymized_id"])).filter(pl.col("anonymized_id").is_in(valid_ids["anonymized_id"]))
/tmp/ipykernel_15057/2419724157.py:77: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  y_train = y_train.filter(~pl.col("anonymized_id").is_in(dup_ids["anonymized_id"])).filter(pl.col("anonymized_id").is_in(valid_ids["anonymized_id"]))


# Z Scaling

We scale X_train and apply it to y_train.

In [6]:
# compute z-score stats per ID + feature (across the sequence dimension)
means = X_train.mean(dim=0, keepdim=True)    
stds  = X_train.std(dim=0, keepdim=True)     

# z-score normalize
X_train = (X_train - means) / stds
y_train = (y_train - means) / stds

# Custom Loss to Use for Positions Given an Order Book

In [ ]:
class CustomExecutionLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, pred_price, close_price, pred_volume, volume_to_fill):
        price_error = torch.abs(pred_price - close_price) / close_price
        total_volume_executed = torch.sum(pred_volume, dim=-1)
        volume_factor = torch.minimum(
            torch.tensor(100.0, device=pred_price.device),
            volume_to_fill / (total_volume_executed + 1e-8)
        )
        loss = price_error * volume_factor
        return torch.mean(loss)

# DeepLOB Encoder

In [ ]:
class deeplob_encoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        # convolution blocks
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(1,2), stride=(1,2)),
            nn.LeakyReLU(negative_slope=0.01),
#             nn.Tanh(),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4,1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4,1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(1,2), stride=(1,2)),
            nn.Tanh(),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4,1)),
            nn.Tanh(),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4,1)),
            nn.Tanh(),
            nn.BatchNorm2d(32),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(1,5)), # changed from (1, 10) to (1, 5) because we have 5 levels instead of 10.
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4,1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4,1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
        )
        
        # inception moduels
        self.inp1 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3,1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(5,1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3, 1), stride=(1, 1), padding=(1, 0)),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
        )
        
        # lstm layers
        self.lstm = nn.LSTM(input_size=192, hidden_size=64, num_layers=1, batch_first=True)

    def forward(self, x):
        # h0: (number of hidden layers, batch size, hidden size)
        h0 = torch.zeros(1, x.size(0), 64).to(device)
        c0 = torch.zeros(1, x.size(0), 64).to(device)
    
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        
        x_inp1 = self.inp1(x)
        x_inp2 = self.inp2(x)
        x_inp3 = self.inp3(x)  
        
        x = torch.cat((x_inp1, x_inp2, x_inp3), dim=1)
        
#         x = torch.transpose(x, 1, 2)
        x = x.permute(0, 2, 1, 3)
        x = torch.reshape(x, (-1, x.shape[1], x.shape[2]))
        
        x, _ = self.lstm(x, (h0, c0))
        x = x[:, -1, :]
        
        return x

In [ ]:
with torch.no_grad():
    sample_input = torch.randn(1, 1, 3600-60, 20).to(device)  # example input: 1 hour minus last minute and 5 levels with 4 features each
    model = deeplob_encoder().to(device)
    sample_output = model(sample_input)
    print(f"Sample output shape: {sample_output.shape}")  # should be (1, 64)

# LSTM Encoder-Decoder

In [10]:
class lstm_encoder(nn.Module):
    ''' Encodes time-series sequence '''

    def __init__(self, input_size, hidden_size, num_layers = 1):
        
        '''
        : param input_size:     the number of features in the input X
        : param hidden_size:    the number of features in the hidden state h
        : param num_layers:     number of recurrent layers (i.e., 2 means there are
        :                       2 stacked LSTMs)
        '''
        
        super(lstm_encoder, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # define LSTM layer
        self.lstm = nn.LSTM(input_size = input_size, hidden_size = hidden_size,
                            num_layers = num_layers)

    def forward(self, x_input):
        
        '''
        : param x_input:               input of shape (seq_len, # in batch, input_size)
        : return lstm_out, hidden:     lstm_out gives all the hidden states in the sequence;
        :                              hidden gives the hidden state and cell state for the last
        :                              element in the sequence 
        '''
        
        lstm_out, self.hidden = self.lstm(x_input.view(x_input.shape[0], x_input.shape[1], self.input_size))
        
        return lstm_out, self.hidden     
    
    def init_hidden(self, batch_size):
        
        '''
        initialize hidden state
        : param batch_size:    x_input.shape[1]
        : return:              zeroed hidden state and cell state 
        '''
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))


class lstm_decoder(nn.Module):
    ''' Decodes hidden state output by encoder '''
    
    def __init__(self, input_size, hidden_size, num_layers = 1):

        '''
        : param input_size:     the number of features in the input X
        : param hidden_size:    the number of features in the hidden state h
        : param num_layers:     number of recurrent layers (i.e., 2 means there are
        :                       2 stacked LSTMs)
        '''
        
        super(lstm_decoder, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size = input_size, hidden_size = hidden_size,
                            num_layers = num_layers)
        self.linear = nn.Linear(hidden_size, input_size)           

    def forward(self, x_input, encoder_hidden_states):
        
        '''        
        : param x_input:                    should be 2D (batch_size, input_size)
        : param encoder_hidden_states:      hidden states
        : return output, hidden:            output gives all the hidden states in the sequence;
        :                                   hidden gives the hidden state and cell state for the last
        :                                   element in the sequence 
 
        '''
        
        lstm_out, self.hidden = self.lstm(x_input.unsqueeze(0), encoder_hidden_states)
        output = self.linear(lstm_out.squeeze(0))     
        
        return output, self.hidden

class lstm_seq2seq(nn.Module):
    ''' train LSTM encoder-decoder and make predictions.
     we want to add the encoded lob data here. '''
    
    def __init__(self, input_size, hidden_size):

        '''
        : param input_size:     the number of expected features in the input X
        : param hidden_size:    the number of features in the hidden state h
        '''

        super(lstm_seq2seq, self).__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size

        self.encoder = lstm_encoder(input_size = input_size, hidden_size = hidden_size)
        self.decoder = lstm_decoder(input_size = input_size, hidden_size = hidden_size)


    def train_model(self, input_tensor, target_tensor, n_epochs, target_len, batch_size, training_prediction = 'recursive', teacher_forcing_ratio = 0.5, learning_rate = 0.01, dynamic_tf = False):
        
        '''
        train lstm encoder-decoder
        
        : param input_tensor:              input data with shape (seq_len, # of order books, number features); PyTorch tensor    
        : param target_tensor:             target data with shape (seq_len, # of order books, number features); PyTorch tensor
        : param n_epochs:                  number of epochs 
        : param target_len:                number of values to predict 
        : param batch_size:                number of samples per gradient update
        : param training_prediction:       type of prediction to make during training ('recursive', 'teacher_forcing', or
        :                                  'mixed_teacher_forcing'); default is 'recursive'
        : param teacher_forcing_ratio:     float [0, 1) indicating how much teacher forcing to use when
        :                                  training_prediction = 'teacher_forcing.' For each batch in training, we generate a random
        :                                  number. If the random number is less than teacher_forcing_ratio, we use teacher forcing.
        :                                  Otherwise, we predict recursively. If teacher_forcing_ratio = 1, we train only using
        :                                  teacher forcing.
        : param learning_rate:             float >= 0; learning rate
        : param dynamic_tf:                use dynamic teacher forcing (True/False); dynamic teacher forcing
        :                                  reduces the amount of teacher forcing for each epoch
        : return losses:                   array of loss function for each epoch
        '''
        
        # initialize array of losses 
        losses = np.full(n_epochs, np.nan)

        optimizer = optim.Adam(self.parameters(), lr = learning_rate)
        criterion = nn.MSELoss()

        # calculate number of batch iterations
        # n_batches = int(input_tensor.shape[1] / batch_size)
        n_batches = (input_tensor.shape[1] + batch_size - 1) // batch_size # ceiling division

        with trange(n_epochs) as tr:
            for it in tr:
                
                batch_loss = 0.
                batch_loss_tf = 0.
                batch_loss_no_tf = 0.
                num_tf = 0
                num_no_tf = 0

                for b in range(n_batches):
                    # select data 
                    start = b * batch_size
                    end = min(start + batch_size, input_tensor.shape[1])
                    input_batch = input_tensor[:, start:end, :]
                    target_batch = target_tensor[:, start:end, :]
                    
                    actual_batch_size = end - start

                    # outputs tensor
                    outputs = torch.zeros(target_len, actual_batch_size, input_batch.shape[2]).to(input_tensor.device)

                    # initialize hidden state
                    encoder_hidden = self.encoder.init_hidden(actual_batch_size)

                    # zero the gradient
                    optimizer.zero_grad()

                    # encoder outputs
                    encoder_output, encoder_hidden = self.encoder(input_batch)

                    # decoder with teacher forcing
                    decoder_input = input_batch[-1, :, :]   # shape: (batch_size, input_size)
                    decoder_hidden = encoder_hidden

                    if training_prediction == 'recursive':
                        # predict recursively
                        for t in range(target_len): 
                            decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden)
                            outputs[t] = decoder_output
                            decoder_input = decoder_output

                    if training_prediction == 'teacher_forcing':
                        # use teacher forcing
                        if random.random() < teacher_forcing_ratio:
                            for t in range(target_len): 
                                decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden)
                                outputs[t] = decoder_output
                                decoder_input = target_batch[t, :, :]

                        # predict recursively 
                        else:
                            for t in range(target_len): 
                                decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden)
                                outputs[t] = decoder_output
                                decoder_input = decoder_output

                    if training_prediction == 'mixed_teacher_forcing':
                        # predict using mixed teacher forcing
                        for t in range(target_len):
                            decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden)
                            outputs[t] = decoder_output
                            
                            # predict with teacher forcing
                            if random.random() < teacher_forcing_ratio:
                                decoder_input = target_batch[t, :, :]
                            
                            # predict recursively 
                            else:
                                decoder_input = decoder_output

                    # compute the loss 
                    loss = criterion(outputs, target_batch)
                    batch_loss += loss.item()
                    
                    # backpropagation
                    loss.backward()
                    optimizer.step()

                # loss for epoch 
                batch_loss /= n_batches 
                losses[it] = batch_loss

                # dynamic teacher forcing
                if dynamic_tf and teacher_forcing_ratio > 0:
                    teacher_forcing_ratio = teacher_forcing_ratio - 0.02 

                # progress bar 
                tr.set_postfix(loss="{0:.3f}".format(batch_loss))
                    
        return losses

    def predict(self, input_tensor, target_len):
        
        '''
        : param input_tensor:      input data (seq_len, input_size); PyTorch tensor 
        : param target_len:        number of target values to predict 
        : return np_outputs:       np.array containing predicted values; prediction done recursively 
        '''

        # encode input_tensor
        input_tensor = input_tensor.unsqueeze(1)     # add in batch size of 1
        encoder_output, encoder_hidden = self.encoder(input_tensor)

        # initialize tensor for predictions
        outputs = torch.zeros(target_len, input_tensor.shape[2])

        # decode input_tensor
        decoder_input = input_tensor[-1, :, :]
        decoder_hidden = encoder_hidden
        
        for t in range(target_len):
            decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden)
            outputs[t] = decoder_output.squeeze(0)
            decoder_input = decoder_output
            
        np_outputs = outputs.detach().numpy()
        
        return np_outputs

## LSTM Training

In [13]:
ow = 60 # we want to predict 60 positions

# specify model parameters and train
model = lstm_seq2seq(input_size = X_train.shape[2], hidden_size = 15).to(device)
loss = model.train_model(X_train, y_train, n_epochs = 50, target_len = ow, batch_size = 5, training_prediction = 'mixed_teacher_forcing', teacher_forcing_ratio = 0.6, learning_rate = 0.01, dynamic_tf = False)
# loss = model.train_model(X_train, y_train, n_epochs = 1, target_len = ow, batch_size = 674, training_prediction = 'mixed_teacher_forcing', teacher_forcing_ratio = 0.6, learning_rate = 0.01, dynamic_tf = False)


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [07:26<00:00,  8.93s/it, loss=0.516]


# 

# Predict LOBs

In [14]:


with torch.no_grad():
    model.predict(X_train, target_len = ow)


RuntimeError: shape '[3540, 1, 25]' is invalid for input of size 59649000